In [0]:
%sql
-- View complete version history for agents table
DESCRIBE HISTORY real_estate_dev.bronze.agents;

In [0]:
%sql
DESCRIBE HISTORY real_estate_dev.bronze.sales_transactions_full;

In [0]:
%sql
DESCRIBE HISTORY real_estate_dev.bronze.locations;

In [0]:
%sql DESCRIBE HISTORY real_estate_dev.bronze.properties_full;


In [0]:
# Get version history for all bronze tables
tables = ["agents", "locations", "properties_full", "sales_transactions_full"]

for table in tables:
    print(f"\n{'='*60}")
    print(f"Version History for {table}:")
    print(f"{'='*60}")
    history = spark.sql(f"DESCRIBE HISTORY real_estate_dev.bronze.{table}")
    history.select("version", "timestamp", "operation", "operationMetrics.numFiles", 
                   "operationMetrics.numOutputRows").show(truncate=False)

In [0]:
%sql
-- Query a specific version (replace version number as needed)
SELECT COUNT(*) as row_count, 'version 0' as version_label
FROM real_estate_dev.bronze.agents VERSION AS OF 0

UNION ALL

SELECT COUNT(*) as row_count, 'current version' as version_label
FROM real_estate_dev.bronze.agents

In [0]:
# Compare schema changes across versions
table_name = "agents"
history_df = spark.sql(f"DESCRIBE HISTORY real_estate_dev.bronze.{table_name}")

versions = history_df.select("version").limit(5).collect()

print(f"Schema evolution for {table_name}:")
print("="*80)

for row in versions:
    version = row.version
    df_version = spark.sql(f"SELECT * FROM real_estate_dev.bronze.{table_name} VERSION AS OF {version} LIMIT 0")
    print(f"\nVersion {version}:")
    print(f"  Columns: {df_version.columns}")
    print(f"  Column count: {len(df_version.columns)}")